In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import brain_probability_flux.analyzer.hierarchical_clustering as hc
import brain_probability_flux.plotting as bpp

In [ ]:
# Results are read from data/. To plot a fresh run, set DATA_DIR to its run
# directory, for example Path("../data/2026-10-20").
DATA_DIR = bpp.DATA_DIR

In [ ]:
save_dir = "../figures/fig-01"
max_fig_width = bpp.FIG_WIDTH
max_fig_height = bpp.FIG_HEIGHT
# Gap between panels in the composite figure (pt); the row-1 widths below
# are balanced against it.
GAP_H = bpp.PANEL_GAP_PT

In [ ]:
# Correlation between the 100 parcels over all recordings
correlation = bpp.parcel_correlation(DATA_DIR)

# Fig. 1a

In [ ]:
fh = hc.square_clustermap_height(
    correlation,
    max_fig_width / 4,
    ax_label="Brain parcel",
    matrix_type="precomputed",
)
g, ax, Z = hc.plot_clustermap(
    correlation,
    figsize=(max_fig_width / 4, fh),
    matrix_type="precomputed",
    return_linkage_matrices=True,
    ax_label="Brain parcel",
)
bpp.save_panel(g.figure, save_dir, "a", bbox_inches="tight")

# Fig. 1b

In [ ]:
fig, ax, dn = hc.plot_dendrogram(
    Z,
    no_labels=True,
    threshold=0.70,
    return_dn=True,
    figsize=(max_fig_width / 4, fh),
)
ax.set_xlabel("Brain parcel")
# Headroom above the topmost merge so the "Threshold" legend (loc="best",
# 5pt) has room to sit without overlapping the dendrogram.
ylim_bottom, ylim_top = ax.get_ylim()
ax.set_ylim(ylim_bottom, ylim_top * 1.08)
bpp.save_panel(fig, save_dir, "b", bbox_inches="tight")
cluster_inds = hc.dendrogram_to_cluster_inds(dn)
num_clusters = len(np.unique(cluster_inds))

# Fig. 1c

In [ ]:
surfaces = bpp.brain_surfaces()
cluster_name = f"hierarchical{num_clusters}"
bpp.write_surface_views(
    surfaces, cluster_inds, cluster_name, ["../figures/parcellation", save_dir]
)

In [ ]:
# Height (pt) of the panel label between panels c and d in the composite
label_d_h_pt = 9.19921875

# Panel c/d height: panel-a height minus label-d gap, divided by 2, so
# c+d combined matches panel a/b's height.
H_cd_pt = (fh * 72 - label_d_h_pt) / 2
H_cd_in = H_cd_pt / 72

# Width for equal physical spacing (s_pt) around and between all four images in 2×2 grid.
# Visible-area aspect ratios: row0 xlim=(60,540)/ylim=(430,50)=480/380, row1 xlim=(-5,~600)/ylim=(450,-20)≈604.5/470
_r_row0 = 480 / 380
_r_row1 = 604.5 / 470
_r_img = (_r_row0 + _r_row1) / 2  # ≈ 1.275
_s_pt = 3.0  # equal gap (pt) around and between images
W_cd_pt = _r_img * H_cd_pt + 3 * _s_pt * (1 - _r_img)
W_cd_in = W_cd_pt / 72

# subplots_adjust fractions for equal physical spacing
surface_grid_spacing = {
    "left": _s_pt / W_cd_pt,
    "right": 1 - _s_pt / W_cd_pt,
    "bottom": _s_pt / H_cd_pt,
    "top": 1 - _s_pt / H_cd_pt,
    "wspace": 2 * _s_pt / (W_cd_pt - 3 * _s_pt),
    "hspace": 2 * _s_pt / (H_cd_pt - 3 * _s_pt),
}


def surface_grid(image_dir, name, label):
    with plt.rc_context({"figure.constrained_layout.use": False}):
        fig, axs = plt.subplots(2, 2, figsize=(W_cd_in, H_cd_in))
    bpp.show_surface_views(axs, image_dir, name)
    fig.subplots_adjust(**surface_grid_spacing)
    bpp.save_panel(fig, save_dir, label, bbox_inches="tight")


surface_grid(save_dir, cluster_name, "c")

# Fig. 1d

In [ ]:
network_ids, network_names = bpp.yeo_networks()
bpp.write_surface_views(
    surfaces,
    network_ids,
    "yeo2011_7network_",
    ["../figures/parcellation"],
    num_colors=8,
    cmap="Dark2",
)
surface_grid("../figures/parcellation", "yeo2011_7network_", "d")

# Fig. 1e

In [ ]:
profile = bpp.network_profile(cluster_inds, network_ids)

# Width = remaining space in row 1 after a, b, c/d (so row 1 == row 2 total width)
_W_r2_pt = max_fig_width * 72  # row 2 = f + g + h = 3 × (max_fig_width/3)
_a_w_pt = bpp.svg_width_pt(Path(f"{save_dir}/a.svg"))
_b_w_pt = bpp.svg_width_pt(Path(f"{save_dir}/b.svg"))
w_e_in = (_W_r2_pt - _a_w_pt - _b_w_pt - W_cd_pt - 3 * GAP_H) / 72
h_e_in = fh  # match panel a height

fig, axs = plt.subplots(1, num_clusters, figsize=(w_e_in, h_e_in))
bpp.plot_network_profile(axs, profile, network_names)
for ax in axs:
    # At most two integer ticks so narrow panels (e.g. cluster 4) do not
    # run their tick labels together.
    ax.xaxis.set_major_locator(plt.MaxNLocator(2, integer=True))
fig.supxlabel("Number of overlapped brain parcels")
fig.suptitle("Cluster")
bpp.save_panel(fig, save_dir, "e", bbox_inches="tight")

Panels f-h show one recording; they need the HCP recordings and are made
by `python scripts/hcp_example_panels.py`.